# Camada Gold: Agregações e KPIs de Negócio
Este notebook lê os dados tratados da camada **Silver** e cria tabelas agregadas e views analíticas na camada **Gold**.

**Objetivo:** Facilitar a conexão com ferramentas de Power BI/Tableau e fornecer métricas rápidas.

### Imports


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from pyspark.sql import functions as F
from pyspark.sql import Window as W
from pyspark.sql.functions import regexp_replace, col, initcap, lower, trim, when, lit, StringType, count, isnan, upper, translate
from pyspark.sql.types import DecimalType, BooleanType
from functools import reduce

### Ambiente

In [0]:
%sql
USE CATALOG projeto;
--DROP DATABASE gold CASCADE; -- Executar se tiver uma gold já criada
CREATE DATABASE IF NOT EXISTS gold;

---
# 1. Tabelas Agregadas (Físicas)
Tabelas Delta otimizadas para performance em dashboards.

### Tabela Gold: `gold.fato_chamados`
Coisas q ela faz:
- Agregação das tabelas com relacionamento (1:1):
  - silver.fato_chamados
  - silver.dim_custos
  - silver.dim_pesquisa_satisfacao

In [0]:
# Fato
df_silver_fato_chamados = spark.table("silver.fato_chamados")

# Dimensões
df_dim_chamados_data = spark.table("silver.dim_chamados_data")
df_dim_pesquisa_satisfacao = spark.table("silver.dim_pesquisa_satisfacao")
df_dim_custos = spark.table("silver.dim_custos")

df_gold = (
    df_silver_fato_chamados
        # Join com dimensão de datas
        .join(
            df_dim_chamados_data,
            on=df_silver_fato_chamados["id_chamado"] == df_dim_chamados_data["id_chamado"],
            how="inner"
        )
        # Join com dimensão de pesquisa de satisfação
        .join(
            df_dim_pesquisa_satisfacao,
            on=df_silver_fato_chamados["id_chamado"] == df_dim_pesquisa_satisfacao["id_chamado"],
            how="left"
        )
        # Join com dimensão de custos
        .join(
            df_dim_custos,
            on=df_silver_fato_chamados["id_chamado"] == df_dim_custos["id_chamado"],
            how="left"
        )
        .select(
            # Todas da fato
            df_silver_fato_chamados["*"],

            # Somente colunas específicas da dimensão de datas
            df_dim_chamados_data["data_hora_abertura"],
            df_dim_chamados_data["data_hora_inicio_atendimento"],
            df_dim_chamados_data["data_hora_finalizacao_atendimento"],
            df_dim_chamados_data["tempo_espera_atendimento_min"],
            df_dim_chamados_data["tempo_atendimento_min"],
            df_dim_chamados_data["ano"],
            df_dim_chamados_data["mes"],
            df_dim_chamados_data["dia"],
            df_dim_chamados_data["dia_semana_num"],
            df_dim_chamados_data["nome_dia"],
            df_dim_chamados_data["trimestre"],
            df_dim_chamados_data["semana_do_ano"],
            df_dim_chamados_data["flag_fim_de_semana"],
            df_dim_chamados_data["ciclo_operacional"],
            df_dim_chamados_data["evento_sazonal"],

            # Colunas específicas das outras dimensões
            df_dim_pesquisa_satisfacao["nota_atendimento"],
            df_dim_custos["valor_custo"]
        )
)

df_gold.write.mode("overwrite").saveAsTable("gold.fato_chamados")

display(df_gold.limit(10))

### Tabela Gold: `gold.desempenho_canal`
Coisas q ela faz:
- Consolida a visão de eficiência técnica e financeira dos canais.
  - Agregação das tabelas com relacionamento:
  - silver.dim_pesquisa_satisfacao
  - silver.dim_canais (Nome do canal)
  - silver.dim_custos (Financeiro)
  - silver.dim_pesquisa_satisfacao (Qualidade)
  - silver.dim_motivos (Criticidade e Contexto)

  

In [0]:
df_chamados = spark.table("projeto.silver.fato_chamados")
df_canais   = spark.table("projeto.silver.dim_canais")
df_custos   = spark.table("projeto.silver.dim_custos")
df_pesquisa = spark.table("projeto.silver.dim_pesquisa_satisfacao")
df_motivos  = spark.table("projeto.silver.dim_motivos")

df_join = (
    df_chamados.alias("c")
        .join(
            df_canais.alias("can"),
            F.lower(F.col("c.nome_canal")) == F.lower(F.col("can.nome_canal")),
            how="left"
        )
        .join(
            df_custos.alias("fin"),
            F.col("c.id_chamado") == F.col("fin.id_chamado"),
            how="left"
        )
        .join(
            df_pesquisa.alias("pesq"),
            F.col("c.id_chamado") == F.col("pesq.id_chamado"),
            how="left"
        )
        .join(
            df_motivos.alias("m"),
            F.lower(F.col("c.nome_motivo_clean")) == F.lower(F.col("m.nome_motivo_clean")),
            how="left"
        )
)

df_gold_desempenho_canal = df_join.select(
    F.col("c.id_chamado"),
    F.col("c.id_cliente"),
    F.col("c.id_atendente"),
    F.col("c.nome_canal"),
    F.col("c.resolvido"),
    F.coalesce(F.col("m.nome_motivo"), F.col("c.nome_motivo_clean")).alias("motivo"),
    F.col("m.categoria_motivo"), 
    F.col("m.criticidade_motivo"),
    F.col("fin.valor_custo"),
    F.col("pesq.nota_atendimento"),
    F.current_timestamp().alias("gold_ingestion_timestamp")
)

display(df_gold_desempenho_canal.limit(20))
df_gold_desempenho_canal.write.mode("overwrite").saveAsTable("projeto.gold.fato_desempenho_canal")

### Tabela Gold: `gold_experiencia_cliente`
Coisas q ela faz:
- Cria a visão 360º da jornada do consumidor enriquecida com segmentação.
- Agregação das tabelas com relacionamento
  - silver.fato_chamados (Base)
  - silver.dim_clientes (Perfil demográfico)
  - silver.dim_pesquisa_satisfacao (Nota do cliente)
  - silver.dim_canais (Canal utilizado)

In [0]:
df_chamados = spark.table("projeto.silver.fato_chamados")
df_clientes = spark.table("projeto.silver.dim_clientes")
df_pesquisa = spark.table("projeto.silver.dim_pesquisa_satisfacao")
df_canais   = spark.table("projeto.silver.dim_canais")

df_join_cliente = df_chamados.alias("c") \
    .join(df_clientes.alias("cli"), on="id_cliente", how="left") \
    .join(df_pesquisa.alias("pesq"), on="id_chamado", how="left") \
    .join(df_canais.alias("can"), F.lower(F.col("c.nome_canal")) == F.lower(F.col("can.nome_canal")), how="left")

df_gold_experiencia_cliente = df_join_cliente.select(
    "c.id_chamado",
    "c.id_cliente",
    "cli.nome",
    "cli.faixa_etaria_geracao", 
    "cli.flag_idoso",
    "cli.regiao",
    F.coalesce("can.nome_canal", "c.nome_canal").alias("canal_utilizado"),
    "c.resolvido",
    "pesq.nota_atendimento",
    F.when(F.col("c.resolvido") == True, 1).otherwise(0).alias("flag_sucesso"),
    F.when(F.col("pesq.nota_atendimento").isNotNull(), 1).otherwise(0).alias("flag_respondeu_pesquisa")
)

df_gold_experiencia_cliente.write.mode("overwrite").saveAsTable("gold.fato_experiencia_cliente")
display(df_gold_experiencia_cliente.limit(5))

### Tabela Gold: `Tabela4`
VCoisas q ela faz:
- coisa 1
- coisa 2
- coisa 3


---
# 2. Views (Camada Lógica)
Views são consultas salvas que não duplicam dados, ideais para recortes específicos de negócio ou "Live Reports".

### View Gold: `gold.view_desempenho_atendente`
Coisas q ela faz:
- coisa 1
- coisa 2
- coisa 3

Desenpenho_atendentes
OBJETIVO: Vizualizar o desempelho dos atendentes
- id_atendente
- nome_atendente
- nivel
- tempo_medio_atendimento
- taxa de resolução de chamadas
- qts_chamadas
- nota média por atendimento

### View Gold: `View2`
Coisas q ela faz:
- coisa 1
- coisa 2
- coisa 3

### View Gold: `View3`
Coisas q ela faz:
- coisa 1
- coisa 2
- coisa 3

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW projeto.gold.vw_canais AS
    SELECT 
        nome_canal,
        COUNT(id_chamado) AS qtd_chamados,
        ROUND(SUM(CASE WHEN resolvido = true THEN 1 ELSE 0 END) / COUNT(id_chamado) * 100, 2) AS taxa_resolucao_pct,
        ROUND(100 - (SUM(CASE WHEN resolvido = true THEN 1 ELSE 0 END) / COUNT(id_chamado) * 100), 2) AS taxa_potencial_transferencia_pct,
        ROUND(AVG(nota_atendimento), 2) AS csat_medio,
        ROUND(SUM(valor_custo), 2) AS custo_total_canal,
        ROUND(AVG(valor_custo), 2) AS custo_medio_por_chamado
        
    FROM projeto.gold.fato_desempenho_canal
    GROUP BY nome_canal
    ORDER BY qtd_chamados DESC
""")

spark.sql("""
    CREATE OR REPLACE VIEW projeto.gold.vw_comparativo_humano_robo AS
    SELECT 
        CASE 
            WHEN upper(nome_canal) IN ('CHATBOT', 'URA', 'whatsapp bot', 'app', 'site') THEN 'Robô/Digital'
            ELSE 'Atendimento Humano' 
        END AS tipo_atendimento,
        
        COUNT(id_chamado) AS volumetria,
        ROUND(AVG(nota_atendimento), 2) AS csat_medio,
        ROUND(SUM(CASE WHEN resolvido = true THEN 1 ELSE 0 END) / COUNT(id_chamado) * 100, 2) AS taxa_resolucao_pct,
        ROUND(SUM(valor_custo), 2) AS custo_total_acumulado,
        ROUND(AVG(valor_custo), 4) AS custo_medio_unitario
    FROM projeto.gold.fato_desempenho_canal
    GROUP BY 1
""")

spark.sql("""
    CREATE OR REPLACE VIEW projeto.gold.vw_diagnostico_bots AS
    SELECT 
        nome_canal,
        criticidade_motivo AS complexidade,
        COUNT(id_chamado) AS tentativas,
        ROUND(SUM(CASE WHEN resolvido = true THEN 1 ELSE 0 END) / COUNT(id_chamado) * 100, 1) AS taxa_sucesso_pct,
        ROUND(AVG(nota_atendimento), 1) AS satisfacao_usuario
        
    FROM projeto.gold.fato_desempenho_canal
    WHERE upper(nome_canal) IN ('URA', 'CHATBOT', 'ATENDIMENTO INICIAL', 'ATENDIMENTO ESPECIALIZADO')
    GROUP BY nome_canal, criticidade_motivo
    ORDER BY nome_canal, tentativas DESC
""")